In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from config import config
from utils import logger, MyLogger
import os
import torch
import torch.nn as nn
import torch.nn.functional as F

os.chdir(config.paths.roots.project)

torch.cuda.empty_cache()
MyLogger.init_loggers()

# Check Data Module

In [ ]:
from src import MatchesDataModule
dm = MatchesDataModule()

dm.setup(stage='fit')

In [ ]:
dl = dm.val_dataloader()

In [ ]:
batch = next(iter(dl))

In [ ]:
reference_patches = batch.reference_patches
target_patches = batch.target_patches
print(reference_patches.shape, target_patches.shape) 

patch_level_reference_coords = batch.patch_level_reference_coords
patch_level_target_coords = batch.patch_level_target_coords
print(patch_level_reference_coords.shape, patch_level_target_coords.shape)

In [ ]:
from utils import show_batch

show_batch(
    reference_patches, target_patches,
    patch_level_reference_coords, patch_level_target_coords,
    limit_count=20, 
    n_columns=4,
)

In [ ]:
# for index, batch in enumerate(dl):
#     print(f'-- Batch Index {index} -----')
#     reference_patches = batch.reference_patches
#     target_patches = batch.target_patches
#     print(reference_patches.shape, target_patches.shape) 

#     patch_level_reference_coords = batch.patch_level_reference_coords
#     patch_level_target_coords = batch.patch_level_target_coords
#     print(patch_level_reference_coords.shape, patch_level_target_coords.shape)


In [ ]:
dm.teardown()

# Check Model

In [ ]:
from src import MatcherModel
model = MatcherModel()

In [ ]:
model.feature_extractor(reference_patches).shape

In [ ]:
print(f'reference_patches shape : {reference_patches.shape}')
print(f'target_patches shape : {target_patches.shape}')
print(f'patch_level_reference_coords shape : {patch_level_reference_coords.shape}')

target_coords_pred = model(reference_patches, target_patches, patch_level_reference_coords)

print(f'target_coords_pred shape : {target_coords_pred.shape}')

In [ ]:
loss = F.mse_loss(
    target_coords_pred.float() / 31.0,
    patch_level_target_coords.float() / 31.0,    
)

loss

In [ ]:
# from torchvision.models import mobilenet_v2
# import torchvision

# feature_extractor = mobilenet_v2(
#     weights=torchvision.models.MobileNet_V2_Weights.IMAGENET1K_V2,
# )

# total_trainable_params = sum(
#     p.numel() for p in feature_extractor.parameters() if p.requires_grad
# )

# total_trainable_params

In [ ]:
model.feature_extractor

# Check Light

In [ ]:
# from src import Light
# light = Light()

In [ ]:
# target_patch_level_coords_pred = light.forward(
#     reference_patches, 
#     target_patches, 
#     patch_level_reference_coords
# )
# 
# print(target_patch_level_coords_pred.shape)